In [1]:
"""
Calcul des scores des modèles sur qcm_final_with_responses.xlsx (QCM, 4 options)
--------------------------------------------------------------------------------
Détecte AUTOMATIQUEMENT toutes les colonnes "response_<modele>" présentes dans
la feuille (sauf response_A/B/C/D qui sont les options, pas des réponses de
modèle, et sauf d'éventuelles colonnes _raw / _attempts encore présentes)
-> aucune modification nécessaire si tu ajoutes de nouveaux modèles, il suffit
de relancer la cellule.

Produit un classeur Excel séparé avec plusieurs feuilles :
  - Notes                  : explication des feuilles et des métriques
  - Dataset                : taille et répartition du jeu de données évalué
  - Overview                : classement des modèles (exactitude, couverture, biais A/B/C/D...)
  - Acc_Axe / N_Axe                    : exactitude / effectifs par Axe
  - Acc_Difficulty / N_Difficulty      : exactitude / effectifs par Difficulty
  - Acc_intent_group / N_intent_group  : exactitude / effectifs par intent_group
  - Acc_intent_category / N_intent_category : exactitude / effectifs par intent_category
  - Confusion               : matrice de confusion A/B/C/D + non-parsé/manquant/autre par modèle
  - Details                 : chaque item + correction (vrai/faux) par modèle

Définitions clés (voir aussi la feuille "Notes" dans le fichier de sortie) :
  - "scored"  = le modèle a répondu une lettre parmi A/B/C/D (ni manquant, ni
    "UNPARSED", ni une valeur inattendue comme "E" ou du texte libre).
  - exactitude / rappel par lettre / biais de position sont calculés
    UNIQUEMENT sur les items "scored" -> ne pas confondre un modèle qui se
    trompe avec un modèle qui n'a simplement pas encore répondu ou n'a pas pu
    être parsé (voir "couverture" pour ça, séparément).
"""

import numpy as np
import pandas as pd
from pathlib import Path

from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

# ============================== CONFIG ==============================

INPUT_PATH = "qcm_final_with_responses.xlsx"
INPUT_SHEET = "final_qcm"   # si absente, la seule feuille du classeur est utilisée automatiquement
OUTPUT_PATH = "qcm_final_scores.xlsx"

GROUND_TRUTH_COL = "final_answer"
ANSWER_LETTERS = ["A", "B", "C", "D"]   # lettres valides d'un QCM

STATUS_COL = "status"                   # ignoré si la colonne n'existe pas
VALID_STATUSES = {"accepted"}           # mets None pour désactiver le filtre par statut

MODEL_COL_PREFIX = "response_"
NON_MODEL_RESPONSE_COLS = {f"response_{l}" for l in ANSWER_LETTERS}
NON_MODEL_SUFFIXES = ("_raw", "_attempts")   # au cas où ces colonnes techniques traînent encore

# colonnes de répartition à croiser avec les modèles (ignorées si absentes de la feuille)
GROUP_COLS = ["Axe", "Difficulty", "intent_group", "intent_category"]
DIFFICULTY_ORDER = ["easy", "medium", "hard"]

# ============================== HELPERS DE DONNÉES ==============================

def normalize_letter(x):
    """Nettoie une valeur de réponse : espaces, casse. Garde NaN tel quel."""
    if pd.isna(x):
        return pd.NA
    return str(x).strip().upper()


def detect_model_columns(df: pd.DataFrame) -> list:
    cols = [
        c for c in df.columns
        if c.startswith(MODEL_COL_PREFIX)
        and c not in NON_MODEL_RESPONSE_COLS
        and not c.endswith(NON_MODEL_SUFFIXES)
    ]
    return sorted(cols)


def model_label(col: str) -> str:
    return col[len(MODEL_COL_PREFIX):]


def order_group_values(group_col: str, series: pd.Series) -> list:
    """Ordonne les valeurs d'une colonne de regroupement pour un affichage lisible :
    ordre logique pour Difficulty, sinon par fréquence décroissante."""
    values = series.dropna().unique().tolist()
    if group_col.lower() == "difficulty":
        lower_map = {str(v).strip().lower(): v for v in values}
        ordered = [lower_map[d] for d in DIFFICULTY_ORDER if d in lower_map]
        remaining = sorted([v for v in values if v not in ordered], key=str)
        return ordered + remaining
    counts = series.value_counts()
    return sorted(values, key=lambda v: (-counts.get(v, 0), str(v)))


# ============================== CALCUL DES MÉTRIQUES ==============================

def load_eligible_data():
    if not Path(INPUT_PATH).exists():
        raise FileNotFoundError(f"Fichier introuvable : {INPUT_PATH}")

    xls = pd.ExcelFile(INPUT_PATH)
    sheet = INPUT_SHEET if INPUT_SHEET in xls.sheet_names else None
    if sheet is None:
        if len(xls.sheet_names) == 1:
            sheet = xls.sheet_names[0]
            print(f"Feuille '{INPUT_SHEET}' introuvable, utilisation de l'unique feuille : '{sheet}'")
        else:
            raise ValueError(
                f"Feuille '{INPUT_SHEET}' introuvable. Feuilles disponibles : {xls.sheet_names}. "
                f"Mets à jour INPUT_SHEET."
            )
    df = pd.read_excel(INPUT_PATH, sheet_name=sheet)

    if GROUND_TRUTH_COL not in df.columns:
        raise ValueError(
            f"Colonne de vérité terrain '{GROUND_TRUTH_COL}' introuvable dans la feuille "
            f"'{sheet}'. Colonnes disponibles : {list(df.columns)}"
        )

    model_cols = detect_model_columns(df)
    if not model_cols:
        raise ValueError(
            f"Aucune colonne de réponse de modèle détectée (préfixe '{MODEL_COL_PREFIX}', "
            f"hors {sorted(NON_MODEL_RESPONSE_COLS)} et suffixes {NON_MODEL_SUFFIXES}). "
            f"Vérifie le nom des colonnes."
        )

    n_before = len(df)

    if STATUS_COL in df.columns and VALID_STATUSES is not None:
        df = df[df[STATUS_COL].isin(VALID_STATUSES)]
        print(f"Filtre statut ({VALID_STATUSES}) : {n_before} -> {len(df)} lignes.")
    else:
        print(f"Pas de filtre par statut appliqué ({len(df)} lignes).")

    n_before_gt = len(df)
    df = df[df[GROUND_TRUTH_COL].notna()]
    if len(df) < n_before_gt:
        print(f"{n_before_gt - len(df)} ligne(s) sans '{GROUND_TRUTH_COL}' exclue(s) (pas encore validées).")

    df = df.reset_index(drop=True)
    print(f"-> {len(df)} items éligibles pour le scoring.")
    print(f"-> {len(model_cols)} modèle(s) détecté(s) : {[model_label(c) for c in model_cols]}")

    return df, model_cols


def compute_overview(df: pd.DataFrame, model_cols: list) -> pd.DataFrame:
    truth = df[GROUND_TRUTH_COL].map(normalize_letter)
    n_total = len(df)
    rows = []

    for col in model_cols:
        name = model_label(col)
        pred = df[col].map(normalize_letter)

        is_scored = pred.isin(ANSWER_LETTERS)
        is_unparsed = pred == "UNPARSED"
        is_missing = pred.isna()
        is_other = (~is_scored) & (~is_unparsed) & (~is_missing)

        n_scored = int(is_scored.sum())
        correct = is_scored & (pred == truth)
        n_correct = int(correct.sum())
        accuracy = n_correct / n_scored if n_scored else np.nan

        row = {
            "modele": name,
            "n_total": n_total,
            "n_scored": n_scored,
            "n_correct": n_correct,
            "couverture": n_scored / n_total if n_total else np.nan,
            "exactitude": accuracy,
        }

        for letter in ANSWER_LETTERS:
            truth_letter = is_scored & (truth == letter)
            recall = (correct & (truth == letter)).sum() / truth_letter.sum() if truth_letter.sum() else np.nan
            row[f"rappel_{letter}"] = recall

        for letter in ANSWER_LETTERS:
            row[f"part_reponses_{letter}"] = (pred == letter).sum() / n_scored if n_scored else np.nan

        row["taux_non_parse"] = int(is_unparsed.sum()) / n_total if n_total else np.nan
        row["taux_manquant"] = int(is_missing.sum()) / n_total if n_total else np.nan
        row["taux_valeur_inattendue"] = int(is_other.sum()) / n_total if n_total else np.nan

        rows.append(row)

    out = pd.DataFrame(rows).sort_values("exactitude", ascending=False, na_position="last").reset_index(drop=True)
    ranks = np.where(out["exactitude"].notna(), np.arange(1, len(out) + 1), np.nan)
    out.insert(0, "rang", ranks)
    return out


def compute_breakdown(df: pd.DataFrame, model_cols: list, group_col: str, model_order: list):
    """Retourne (table_exactitude, table_effectifs), modèles en lignes, valeurs du
    groupe en colonnes."""
    truth = df[GROUND_TRUTH_COL].map(normalize_letter)
    acc_data, n_data = {}, {}

    for col in model_cols:
        name = model_label(col)
        pred = df[col].map(normalize_letter)
        is_scored = pred.isin(ANSWER_LETTERS)
        correct = is_scored & (pred == truth)

        tmp = pd.DataFrame({"group": df[group_col], "scored": is_scored, "correct": correct})
        grouped = tmp.groupby("group", dropna=False)
        n_scored_per_group = grouped["scored"].sum()
        n_correct_per_group = grouped["correct"].sum()
        acc_data[name] = (n_correct_per_group / n_scored_per_group).replace([np.inf, -np.inf], np.nan)
        n_data[name] = n_scored_per_group

    acc_df = pd.DataFrame(acc_data).T.reindex(model_order)
    n_df = pd.DataFrame(n_data).T.reindex(model_order)

    col_order = order_group_values(group_col, df[group_col])
    acc_df = acc_df.reindex(columns=col_order)
    n_df = n_df.reindex(columns=col_order).fillna(0).astype(int)

    return acc_df.reset_index().rename(columns={"index": "modele"}), \
           n_df.reset_index().rename(columns={"index": "modele"})


def compute_confusion(df: pd.DataFrame, model_cols: list, model_order: list) -> pd.DataFrame:
    """Matrice de confusion aplatie : une colonne predit_X_reel_Y par paire de
    lettres (16 colonnes pour un QCM à 4 options), + non_parse / manquant / autre."""
    truth = df[GROUND_TRUTH_COL].map(normalize_letter)
    rows = []
    for col in model_cols:
        name = model_label(col)
        pred = df[col].map(normalize_letter)
        row = {"modele": name}
        for p_letter in ANSWER_LETTERS:
            for t_letter in ANSWER_LETTERS:
                row[f"predit_{p_letter}_reel_{t_letter}"] = int(((pred == p_letter) & (truth == t_letter)).sum())
        is_scored = pred.isin(ANSWER_LETTERS)
        is_unparsed = pred == "UNPARSED"
        is_missing = pred.isna()
        row["non_parse"] = int(is_unparsed.sum())
        row["manquant"] = int(is_missing.sum())
        row["autre"] = int((~is_scored & ~is_unparsed & ~is_missing).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("modele").reindex(model_order).reset_index()


def build_details(df: pd.DataFrame, model_cols: list) -> pd.DataFrame:
    base_cols = [c for c in ["item_id", "item_type", "Axe", "Difficulty", "intent_group", "intent_category",
                              "input", "response_A", "response_B", "response_C", "response_D",
                              "original_key", GROUND_TRUTH_COL, STATUS_COL]
                 if c in df.columns]
    details = df[base_cols + model_cols].copy()
    truth = df[GROUND_TRUTH_COL].map(normalize_letter)

    for col in model_cols:
        name = model_label(col)
        pred = df[col].map(normalize_letter)
        is_scored = pred.isin(ANSWER_LETTERS)
        correct_col = pd.Series(pd.NA, index=df.index, dtype=object)
        correct_col.loc[is_scored] = pred.loc[is_scored] == truth.loc[is_scored]
        details[f"{name}_correct"] = correct_col

    return details


def build_dataset_summary(df: pd.DataFrame, group_cols_present: list) -> pd.DataFrame:
    total = len(df)
    truth = df[GROUND_TRUTH_COL].map(normalize_letter)
    rows = []
    for val in ANSWER_LETTERS:
        n = int((truth == val).sum())
        rows.append({"categorie": "final_answer", "valeur": val, "n": n,
                     "part": n / total if total else np.nan})
    for gc in group_cols_present:
        for val, n in df[gc].value_counts(dropna=False).items():
            rows.append({"categorie": gc, "valeur": str(val), "n": int(n),
                         "part": n / total if total else np.nan})
    out = pd.DataFrame(rows)
    out.insert(0, "n_total_dataset", total)
    return out


# ============================== MISE EN FORME EXCEL ==============================

HEADER_FILL = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", bold=True)


def style_header(ws, n_cols: int):
    for c in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=c)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")


def autofit_columns(ws, df: pd.DataFrame):
    for i, col in enumerate(df.columns, start=1):
        try:
            max_len = max([len(str(col))] + [len(str(v)) for v in df[col].astype(str).tolist()])
        except Exception:
            max_len = len(str(col))
        ws.column_dimensions[get_column_letter(i)].width = min(max_len + 3, 45)


def apply_percent_format(ws, df: pd.DataFrame, percent_cols: list):
    for col_name in percent_cols:
        if col_name not in df.columns:
            continue
        col_idx = df.columns.get_loc(col_name) + 1
        for row in range(2, len(df) + 2):
            ws.cell(row=row, column=col_idx).number_format = "0.0%"


def apply_color_scale(ws, df: pd.DataFrame, cols: list, invert: bool = False):
    if len(df) == 0:
        return
    colors = ("F8696B", "FFEB84", "63BE7B") if not invert else ("63BE7B", "FFEB84", "F8696B")
    for col_name in cols:
        if col_name not in df.columns:
            continue
        col_idx = df.columns.get_loc(col_name) + 1
        letter = get_column_letter(col_idx)
        cell_range = f"{letter}2:{letter}{len(df) + 1}"
        rule = ColorScaleRule(
            start_type="min", start_color=colors[0],
            mid_type="percentile", mid_value=50, mid_color=colors[1],
            end_type="max", end_color=colors[2],
        )
        ws.conditional_formatting.add(cell_range, rule)


def write_sheet(writer, df: pd.DataFrame, sheet_name: str,
                 percent_cols=None, color_scale_cols=None, invert_cols=None):
    df.to_excel(writer, sheet_name=sheet_name, index=False)
    ws = writer.sheets[sheet_name]
    n_rows, n_cols = df.shape
    style_header(ws, n_cols)
    autofit_columns(ws, df)
    if n_rows > 0:
        ws.auto_filter.ref = ws.dimensions
        ws.freeze_panes = "B2"
    if percent_cols:
        apply_percent_format(ws, df, percent_cols)
    if color_scale_cols:
        invert_cols = invert_cols or []
        apply_color_scale(ws, df, [c for c in color_scale_cols if c not in invert_cols], invert=False)
        apply_color_scale(ws, df, [c for c in color_scale_cols if c in invert_cols], invert=True)


def write_notes_sheet(writer, model_cols: list, group_cols_present: list):
    wb = writer.book
    ws = wb.create_sheet("Notes", 0)
    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 100

    def line(row, a, b="", bold=False, size=11):
        ca = ws.cell(row=row, column=1, value=a)
        cb = ws.cell(row=row, column=2, value=b)
        if bold:
            ca.font = Font(bold=True, size=size)
            cb.font = Font(bold=True, size=size)
        ca.alignment = Alignment(wrap_text=True, vertical="top")
        cb.alignment = Alignment(wrap_text=True, vertical="top")

    r = 1
    line(r, "Score des modèles — QCM (A/B/C/D)", bold=True, size=14); r += 2
    line(r, "Modèles détectés", ", ".join(model_label(c) for c in model_cols)); r += 1
    line(r, "Dimensions de répartition", ", ".join(group_cols_present) if group_cols_present else "aucune"); r += 2

    line(r, "Feuille", "Contenu", bold=True); r += 1
    for name, desc in [
        ("Dataset", "Taille du jeu de données scoré et répartition (Axe, Difficulty, ..., équilibre A/B/C/D de la vérité terrain)."),
        ("Overview", "Classement des modèles : exactitude, couverture, rappel par lettre (A/B/C/D), biais de position, taux d'erreurs."),
        ("Acc_<dimension>", "Exactitude de chaque modèle, ventilée par valeur de la dimension (ex: par Axe)."),
        ("N_<dimension>", "Nombre d'items réellement scorés par modèle et par valeur de la dimension (à lire à côté de Acc_<dimension> pour juger la fiabilité d'un pourcentage calculé sur peu d'items)."),
        ("Confusion", "Pour chaque modèle : matrice complète prédit×réel sur A/B/C/D, + non-parsé + manquant + autre."),
        ("Details", "Chaque item avec la réponse de chaque modèle et une colonne <modele>_correct (Vrai/Faux/vide)."),
    ]:
        line(r, name, desc)
        r += 1

    r += 1
    line(r, "Métrique", "Définition", bold=True); r += 1
    for name, desc in [
        ("scored", "Le modèle a répondu une lettre parmi A/B/C/D (ni manquant, ni 'UNPARSED', ni valeur inattendue)."),
        ("couverture", "n_scored / n_total : part des items pour lesquels le modèle a donné une réponse exploitable."),
        ("exactitude", "n_correct / n_scored : calculée UNIQUEMENT sur les items 'scored' (ne pénalise pas deux fois les non-réponses, déjà visibles via couverture/taux_manquant/taux_non_parse)."),
        ("rappel_A / B / C / D", "Parmi les items 'scored' dont la vérité terrain est cette lettre, part correctement retrouvée. Utile pour repérer un modèle qui répond bien globalement mais rate systématiquement une option en particulier."),
        ("part_reponses_A / B / C / D", "Parmi les réponses 'scored', part de chaque lettre donnée par le modèle. Comparer à l'équilibre réel (feuille Dataset) pour détecter un biais de position (ex: le modèle répond souvent A)."),
        ("taux_non_parse", "Part des items où le modèle a répondu mais sans qu'on ait pu en extraire une lettre A-D."),
        ("taux_manquant", "Part des items sans réponse du tout (pas encore évalués, ou échec réseau/API)."),
        ("taux_valeur_inattendue", "Part des items où le modèle a répondu autre chose qu'une lettre A-D et qu'un texte non-parsable classique (ex: une 5e lettre, un chiffre, une phrase)."),
    ]:
        line(r, name, desc)
        r += 1


# ============================== MAIN ==============================

def main():
    df, model_cols = load_eligible_data()
    group_cols_present = [c for c in GROUP_COLS if c in df.columns]
    missing_groups = [c for c in GROUP_COLS if c not in df.columns]
    if missing_groups:
        print(f"(Dimensions absentes de la feuille, ignorées : {missing_groups})")

    overview = compute_overview(df, model_cols)
    model_order = overview["modele"].tolist()

    dataset_summary = build_dataset_summary(df, group_cols_present)
    confusion = compute_confusion(df, model_cols, model_order)
    details = build_details(df, model_cols)

    out_path = Path(OUTPUT_PATH)
    tmp_path = out_path.with_name(out_path.stem + ".tmp" + out_path.suffix)

    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        write_sheet(
            writer, dataset_summary, "Dataset",
            percent_cols=["part"], color_scale_cols=["part"],
        )

        write_sheet(
            writer, overview, "Overview",
            percent_cols=["couverture", "exactitude",
                          "rappel_A", "rappel_B", "rappel_C", "rappel_D",
                          "part_reponses_A", "part_reponses_B", "part_reponses_C", "part_reponses_D",
                          "taux_non_parse", "taux_manquant", "taux_valeur_inattendue"],
            color_scale_cols=["exactitude", "couverture",
                              "rappel_A", "rappel_B", "rappel_C", "rappel_D"],
            invert_cols=[],
        )

        for group_col in group_cols_present:
            acc_df, n_df = compute_breakdown(df, model_cols, group_col, model_order)
            value_cols = [c for c in acc_df.columns if c != "modele"]

            write_sheet(
                writer, acc_df, f"Acc_{group_col}"[:31],
                percent_cols=value_cols, color_scale_cols=value_cols,
            )
            write_sheet(
                writer, n_df, f"N_{group_col}"[:31],
            )

        write_sheet(writer, confusion, "Confusion")
        write_sheet(writer, details, "Details")

        # Notes en tout premier onglet, une fois que toutes les autres feuilles existent
        write_notes_sheet(writer, model_cols, group_cols_present)

    tmp_path.replace(out_path)
    print(f"\nTerminé -> {out_path.resolve()}")


try:
    main()
except Exception as e:
    print(f"\nÉCHEC : {e}")
    raise


Filtre statut ({'accepted'}) : 229 -> 229 lignes.
-> 229 items éligibles pour le scoring.
-> 12 modèle(s) détecté(s) : ['claude_sonnet5', 'commandr', 'gemini3_6_flash', 'gemma4_26b', 'gemma4_31b', 'gpt5', 'gptoss_20b', 'grok4_20', 'mistral_24b', 'mistral_saba', 'qwen3_5_9b', 'qwen3_6_27b']

Terminé -> C:\Users\Abderrahman\Desktop\moderation llm\paper redac 27 juillet 2026\final_binary_scores.xlsx
